In [0]:
import requests
import base64
from urllib.parse import quote


organization = "ozanozeer"
project = "aXet Project"
wiki = "aXet-Project.wiki"


pat = dbutils.secrets.get(
    scope="sql-ozoezer",
    key="devopspac"
)


auth = base64.b64encode(
    f":{pat}".encode()
).decode()


headers = {
    "Authorization": f"Basic {auth}",
    "Content-Type": "application/json"
}


def get_wiki_page(page_path):

    project_encoded = quote(project)

    url = (
        f"https://dev.azure.com/{organization}/"
        f"{project_encoded}/_apis/wiki/wikis/"
        f"{wiki}/pages"
        f"?path={page_path}"
        f"&api-version=7.1"
    )

    response = requests.get(
        url,
        headers=headers
    )

    return response


def create_wiki_page(page_path, content):

    project_encoded = quote(project)

    url = (
        f"https://dev.azure.com/{organization}/"
        f"{project_encoded}/_apis/wiki/wikis/"
        f"{wiki}/pages"
        f"?path={page_path}"
        f"&api-version=7.1"
    )

    response = requests.put(
        url,
        headers=headers,
        json={
            "content": content
        }
    )

    return response


def update_wiki_page(page_path, content, version):

    project_encoded = quote(project)

    url = (
        f"https://dev.azure.com/{organization}/"
        f"{project_encoded}/_apis/wiki/wikis/"
        f"{wiki}/pages"
        f"?path={page_path}"
        f"&api-version=7.1"
    )

    headers_update = headers.copy()

    headers_update["If-Match"] = version

    response = requests.put(
        url,
        headers=headers_update,
        json={
            "content": content
        }
    )

    return response


def push_wiki_page(page_path, content):

    existing_page = get_wiki_page(page_path)

    if existing_page.status_code == 200:

        # Azure DevOps ETag header'dan alınır
        version = existing_page.headers.get("ETag")

        if version is None:
            print("ETag bulunamadı")
            print(existing_page.headers)
            return existing_page

        response = update_wiki_page(
            page_path,
            content,
            version
        )

        print("Wiki page updated")

    else:

        response = create_wiki_page(
            page_path,
            content
        )

        print("Wiki page created")


    print(response.status_code)
    print(response.text)

    return response